# Week 2 — Model Packaging & Containerization

### What a container actually is, and how to bundle a model so it runs anywhere

> **MLOps & Deployment: A From-Scratch Engineering Codex**
> Part of the applied-systems track. Like the rest of this curriculum, every
> abstraction here is *built before it is used*: we re-implement the core of each
> MLOps tool in plain Python/NumPy first, then map what we built onto the
> industry-standard equivalent so the magic is gone by the time you reach it.

---

## Learning objectives

By the end of this notebook you will be able to:

1. Articulate the 'dependency hell' problem packaging solves.
2. Design a self-describing model package format (model + env + code + signature).
3. Build a tiny package-and-load workflow that round-trips a trained model.
4. Understand containers as the kernel-level generalisation of your package (namespaces, layers, images).
5. Author and reason about a production-quality Dockerfile for an ML service.

## Prerequisites

- Week 1 (a registered model + run manifest).
- Comfort with the filesystem and JSON; conceptual familiarity with the OS process model helps.

## How to read this notebook

Each section has three layers:

- **🧠 Theory** — the systems problem and *why* it exists.
- **🔧 From scratch** — a minimal, dependency-light implementation you fully own.
- **🏭 In practice** — how the real tool (MLflow, Docker, K8s, etc.) does the same thing, and what it adds.

Run cells top-to-bottom. Everything is self-contained and works on a CPU-only laptop.

In [1]:
# --- Standard setup ---------------------------------------------------------
from __future__ import annotations
import os, sys, json, time, math, hashlib, random, pathlib, tempfile, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Callable

import numpy as np

np.random.seed(0)
random.seed(0)

# A scratch workspace that we clean up between runs of this notebook.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="mlops_week_"))
print("Working directory for this notebook:", WORK)

# --- Content-addressable hashing (introduced & explained in Week 0) ---------
# Re-defined here so each notebook is fully self-contained and runnable alone.
def hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj) -> str:
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    return hash_bytes(a.tobytes() + str(a.shape).encode() + str(a.dtype).encode())

def capture_environment() -> dict:
    import platform
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__}

Working directory for this notebook: /tmp/mlops_week_t2h18_8e


## 1. 🧠 The portability problem

You trained a model on your laptop. It needs to run on a colleague's machine, a
CI runner, and a production server — each with different OS versions, Python
versions, and installed libraries. The model weights alone are useless without:

- the **code** that knows how to load them and run a forward pass,
- the **environment** (exact library versions) they were trained against,
- a **contract** describing the expected input/output shape (the *signature*),
- **metadata**: which run produced this, what metrics, what data hash.

Packaging is the act of bundling all of that into one portable, self-describing
artifact. Containerization takes it one level deeper — bundling the *operating
system userland too* — so the unit is reproducible down to `libc`.

We'll build the package layer ourselves, then explain containers as its natural
extension.

## 2. 🔧 From scratch: a self-describing model package

Real formats (MLflow's `MLmodel`, ONNX, TorchScript bundles) all share a shape: a
directory with the weights, a manifest describing how to load them, and an
environment spec. Let's define our own — call it the `.mlpkg`.

First we need a trivial but *real* model to package. We'll reuse the linear model
idea but wrap it in a clean predict interface.

In [2]:
class LinearModel:
    """A minimal model with a clean, serialisable interface."""
    def __init__(self, w: np.ndarray, b: float):
        self.w = np.asarray(w, dtype=float)
        self.b = float(b)

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.atleast_2d(np.asarray(X, dtype=float))
        return X @ self.w + self.b

    # --- serialisation: weights as plain JSON-able state ---
    def state(self) -> dict:
        return {"w": self.w.tolist(), "b": self.b}

    @classmethod
    def from_state(cls, s: dict) -> "LinearModel":
        return cls(np.array(s["w"]), s["b"])


# "Train" one quickly so we have something to package
rng = np.random.RandomState(0)
Xtr = rng.rand(300, 2)
ytr = Xtr @ np.array([2.0, -1.0]) + 0.5
# closed-form least squares for a clean fit
A = np.hstack([Xtr, np.ones((len(Xtr), 1))])
coef, *_ = np.linalg.lstsq(A, ytr, rcond=None)
model = LinearModel(coef[:2], coef[2])
print("Trained weights:", model.w, "bias:", round(model.b, 4))
print("Sanity prediction for [0.5, 0.5]:", model.predict([0.5, 0.5]))

Trained weights: [ 2. -1.] bias: 0.5
Sanity prediction for [0.5, 0.5]: [1.]


### 2.1 The package format

An `.mlpkg` is just a directory:

```
model.mlpkg/
├── MLPKG.json        # the manifest: how to load, signature, metadata
├── weights.json      # the model state
└── predict.py        # the code that turns state -> a callable
```

The manifest is the heart of it. Anyone — any language, any tool — can read
`MLPKG.json` and know how to use the package without guessing. This *self-describing*
property is what makes the artifact portable.

In [3]:
def infer_signature(X_sample: np.ndarray, y_sample: np.ndarray) -> dict:
    """Capture the input/output contract. Mismatches here are the #1 serving bug."""
    return {
        "input":  {"shape": [None, int(X_sample.shape[1])], "dtype": str(X_sample.dtype)},
        "output": {"shape": [None], "dtype": str(np.asarray(y_sample).dtype)},
    }

PREDICT_CODE = '''
import json, numpy as np
def load(pkg_dir):
    state = json.load(open(pkg_dir + "/weights.json"))
    w = np.array(state["w"]); b = float(state["b"])
    def predict(X):
        X = np.atleast_2d(np.asarray(X, dtype=float))
        return X @ w + b
    return predict
'''

def save_package(model: LinearModel, path: pathlib.Path, signature: dict, metadata: dict):
    path = pathlib.Path(path); path.mkdir(parents=True, exist_ok=True)
    (path / "weights.json").write_text(json.dumps(model.state()))
    (path / "predict.py").write_text(PREDICT_CODE)
    manifest = {
        "format": "mlpkg/1",
        "flavor": "python_function",
        "loader": "predict.load",          # module.function to call
        "artifacts": ["weights.json", "predict.py"],
        "signature": signature,
        "env": capture_environment_min(),
        "metadata": metadata,
        # content hash of weights = the artifact's identity (Week 0 idea!)
        "weights_sha": hashlib.sha256((path / "weights.json").read_bytes()).hexdigest()[:12],
    }
    (path / "MLPKG.json").write_text(json.dumps(manifest, indent=2))
    return path

def capture_environment_min() -> dict:
    return {"python": sys.version.split()[0], "numpy": np.__version__}

sig = infer_signature(Xtr, ytr)
pkg_path = save_package(
    model, WORK / "model.mlpkg", sig,
    metadata={"run_id": "wk1-champion", "task": "regression", "data_sha": "demo"},
)
print("Package written to:", pkg_path)
print("\nMLPKG.json:\n", (pkg_path / "MLPKG.json").read_text())

Package written to: /tmp/mlops_week_t2h18_8e/model.mlpkg

MLPKG.json:
 {
  "format": "mlpkg/1",
  "flavor": "python_function",
  "loader": "predict.load",
  "artifacts": [
    "weights.json",
    "predict.py"
  ],
  "signature": {
    "input": {
      "shape": [
        null,
        2
      ],
      "dtype": "float64"
    },
    "output": {
      "shape": [
        null
      ],
      "dtype": "float64"
    }
  },
  "env": {
    "python": "3.12.3",
    "numpy": "2.4.4"
  },
  "metadata": {
    "run_id": "wk1-champion",
    "task": "regression",
    "data_sha": "demo"
  },
  "weights_sha": "ae5405c99566"
}


## 3. 🔧 Loading a package in a fresh process

The real test of portability: can a process that *knows nothing about
`LinearModel`* load and run this package using only the manifest? Yes — because
the manifest tells it exactly how. This is dynamic loading, the same mechanism
MLflow's `pyfunc.load_model` uses.

In [4]:
import importlib.util

def load_package(path: pathlib.Path):
    path = pathlib.Path(path)
    manifest = json.loads((path / "MLPKG.json").read_text())
    module_name, func_name = manifest["loader"].split(".")
    # Dynamically import predict.py without it being on sys.path
    spec = importlib.util.spec_from_file_location(module_name, path / f"{module_name}.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    loader_fn = getattr(mod, func_name)
    predict = loader_fn(str(path))
    return predict, manifest

predict_fn, manifest = load_package(pkg_path)
# Validate against the declared signature before trusting the model
def validate_input(X, manifest):
    expected_dim = manifest["signature"]["input"]["shape"][1]
    X = np.atleast_2d(np.asarray(X))
    if X.shape[1] != expected_dim:
        raise ValueError(f"signature mismatch: expected {expected_dim} features, got {X.shape[1]}")
    return X

X_new = validate_input([[0.5, 0.5], [0.1, 0.9]], manifest)
print("Predictions from reloaded package:", predict_fn(X_new))
print("Matches original model?",
      np.allclose(predict_fn(X_new), model.predict(X_new)))

# Show the signature guard actually fires
try:
    validate_input([[0.5, 0.5, 0.5]], manifest)
except ValueError as e:
    print("Signature guard caught bad input:", e)

Predictions from reloaded package: [ 1.  -0.2]
Matches original model? True
Signature guard caught bad input: signature mismatch: expected 2 features, got 3


That signature check is not academic. The single most common production-serving
bug is *training/serving skew*: the model expects features in one shape/order and
the live request sends another. The package carries its own contract so the
mismatch is caught at the door, not 10,000 wrong predictions later.

## 4. 🧠 From package to container: the next layer down

Our `.mlpkg` bundles **code + weights + python/numpy versions**. But it still
assumes the host has *a* Python, *a* libc, *a* set of system libraries. On a
machine with a different OpenSSL or glibc, your numpy wheel might not even import.

A **container** closes that gap by bundling the entire userland — every shared
library, the package manager, the interpreter — everything except the Linux
kernel itself. It's the same idea as our package, pushed down to the OS level.

Two kernel features make containers possible (and they are *not* virtual machines):

- **Namespaces** — give a process its own isolated view of the filesystem, network,
  process tree, users. The process thinks it's alone on the machine.
- **cgroups** — limit and account for resources (CPU, memory) per group of processes.

A container is just *a normal Linux process* running with its own namespaces and
cgroups, using a filesystem image you provide. There is no hypervisor; that's why
containers start in milliseconds.

### 4.1 🔧 The layered-image idea, in Python

A Docker image is a stack of read-only **layers**, each identified by a content
hash (Week 0 again!). Building reuses unchanged layers from cache. We can model
the essential idea — content-addressed, cache-reusing layers — without Docker.

In [5]:
@dataclass
class Layer:
    instruction: str          # e.g. "COPY weights.json"
    content_hash: str         # hash of (parent_hash + instruction + payload)

class ImageBuilder:
    """Models Docker's layer caching: identical instruction stacks reuse layers."""
    def __init__(self):
        self.cache: dict[str, Layer] = {}   # content_hash -> Layer
        self.builds = 0

    def build(self, instructions: list[tuple[str, bytes]]) -> list[Layer]:
        layers, parent = [], "scratch"
        for instr, payload in instructions:
            h = hashlib.sha256(parent.encode() + instr.encode() + payload).hexdigest()[:10]
            if h in self.cache:
                layer = self.cache[h]            # CACHE HIT — reused
            else:
                self.builds += 1
                layer = Layer(instr, h)
                self.cache[h] = layer            # CACHE MISS — built once
            layers.append(layer)
            parent = h
        return layers

builder = ImageBuilder()
recipe = [
    ("FROM python:3.11-slim", b""),
    ("RUN pip install numpy", b"numpy==1.26"),
    ("COPY model.mlpkg", (pkg_path / "weights.json").read_bytes()),
    ("CMD python serve.py", b""),
]
print("First build:")
for L in builder.build(recipe):
    print("  layer", L.content_hash, "-", L.instruction)
print("layers actually built:", builder.builds)

# Rebuild after changing only the LAST instruction -> earlier layers are reused
recipe2 = recipe[:-1] + [("CMD python serve.py --workers 4", b"")]
print("\nRebuild with changed CMD:")
before = builder.builds
for L in builder.build(recipe2):
    print("  layer", L.content_hash, "-", L.instruction)
print("new layers built this time:", builder.builds - before, "(only the changed tail)")

First build:
  layer cfcdae4d74 - FROM python:3.11-slim
  layer 3e82d0d0ba - RUN pip install numpy
  layer 6b8171b234 - COPY model.mlpkg
  layer f669435e99 - CMD python serve.py
layers actually built: 4

Rebuild with changed CMD:
  layer cfcdae4d74 - FROM python:3.11-slim
  layer 3e82d0d0ba - RUN pip install numpy
  layer 6b8171b234 - COPY model.mlpkg
  layer 3ccc07b272 - CMD python serve.py --workers 4
new layers built this time: 1 (only the changed tail)


This is *exactly* why Dockerfile instruction order matters: put the things that
change rarely (base image, system deps) early, and the things that change every
commit (your code) late. Then 90% of your builds reuse the expensive layers. You
just implemented the caching logic that makes that true.

## 5. 🏭 In practice: a real Dockerfile for an ML service

Here is a production-shaped Dockerfile for serving our package. Read every line —
each one maps to a concept we built.

In [6]:
DOCKERFILE = '''
# ---- Stage 1: builder (heavy, throwaway) ----------------------------------
FROM python:3.11-slim AS builder
WORKDIR /build
# Copy ONLY the dependency spec first so this layer caches across code changes
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# ---- Stage 2: runtime (small, what we actually ship) ----------------------
FROM python:3.11-slim AS runtime
# Run as a non-root user: a container escape shouldn't be a root shell
RUN useradd --create-home appuser
WORKDIR /app
# Bring over ONLY the installed packages from the builder (multi-stage = small image)
COPY --from=builder /install /usr/local
# Copy the model package and serving code (changes most often -> latest layer)
COPY model.mlpkg/ ./model.mlpkg/
COPY serve.py .
USER appuser
EXPOSE 8080
# Healthcheck lets the orchestrator know if we're alive (Week 3 + K8s)
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8080/health')"
CMD ["python", "serve.py"]
'''
(WORK / "Dockerfile").write_text(DOCKERFILE)
print(DOCKERFILE)


# ---- Stage 1: builder (heavy, throwaway) ----------------------------------
FROM python:3.11-slim AS builder
WORKDIR /build
# Copy ONLY the dependency spec first so this layer caches across code changes
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# ---- Stage 2: runtime (small, what we actually ship) ----------------------
FROM python:3.11-slim AS runtime
# Run as a non-root user: a container escape shouldn't be a root shell
RUN useradd --create-home appuser
WORKDIR /app
# Bring over ONLY the installed packages from the builder (multi-stage = small image)
COPY --from=builder /install /usr/local
# Copy the model package and serving code (changes most often -> latest layer)
COPY model.mlpkg/ ./model.mlpkg/
COPY serve.py .
USER appuser
EXPOSE 8080
# Healthcheck lets the orchestrator know if we're alive (Week 3 + K8s)
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:80

Why each choice matters, mapped to what we built:

- **Multi-stage build** — the builder stage compiles/installs deps; the runtime
  stage copies only the *results*. Final image is small. (Layer reuse, taken further.)
- **`COPY requirements.txt` before code** — pins the expensive `pip install` layer
  so code edits don't reinstall numpy. (The caching law you just proved.)
- **non-root `appuser`** — least privilege; a compromised process isn't root.
- **`HEALTHCHECK`** — the orchestrator's signature/liveness contract, the OS-level
  cousin of our `validate_input` guard.
- **`EXPOSE 8080`** — declares the serving port; we light it up in Week 3.

> Build & run (on a machine with Docker — not in this notebook):
> ```bash
> docker build -t linreg-service:v1 .
> docker run -p 8080:8080 linreg-service:v1
> ```

## 6. 🏭 Mapping table

| You built | Real-world equivalent |
|-----------|----------------------|
| `.mlpkg` directory + `MLPKG.json` | MLflow `MLmodel`, ONNX, TorchScript bundle |
| `infer_signature` + `validate_input` | MLflow model signatures, Pydantic request schemas |
| `load_package` dynamic import | `mlflow.pyfunc.load_model` |
| `ImageBuilder` content-hashed layers | Docker image layers + build cache |
| `weights_sha` in manifest | image digests, model checksums |
| Multi-stage Dockerfile | the actual standard for slim ML images |

The lesson: a container is not magic. It's your self-describing package, pushed
down to wrap the OS userland, isolated with kernel namespaces, and addressed by
content hash — all ideas you now own in code.

---

## 🎯 Exercises

Try these before moving on. They are deliberately open-ended — the goal is to
extend the machinery you just built, not to find a single right answer.

1. Extend `MLPKG.json` with a `conda_env` / `requirements` field listing exact pins, then write `check_env(manifest)` that warns if the *current* numpy version differs from the packaged one. This is MLflow's env-mismatch warning.
2. Add a `predict.py` that returns prediction *and* a confidence/uncertainty estimate. Update the signature. What changes in the serving contract?
3. Reorder the Dockerfile to put `COPY serve.py` before `COPY requirements.txt`. Using the `ImageBuilder` model, predict how many layers rebuild on a code change vs the correct ordering. Verify with the simulator.
4. Our `ImageBuilder` keys a layer on (parent + instruction + payload). Show that two builds that install deps in a *different order* produce different hashes even if the final environment is identical. Why is this both a feature (reproducibility) and a footgun (cache misses)?
5. Write `bundle_to_tar(pkg_path)` that turns the `.mlpkg` directory into a single `.tar.gz` artifact with its `weights_sha` in the filename. This is how registries store immutable, downloadable model bundles.

---

## ✅ Recap — Week 2

- Packaging bundles model + code + env + signature + metadata into one self-describing artifact.
- A model signature is the input/output contract that prevents training/serving skew.
- Dynamic loading from a manifest is how a model runs in a process that never imported its class.
- A container is a normal Linux process with its own namespaces/cgroups + a content-hashed layered filesystem — not a VM.
- Dockerfile instruction order is layer-cache strategy; put stable things early, volatile things late.

### Coming up next

**Week 3 — Serving & Inference.** Our packaged model is portable but inert. We'll build an inference server from raw sockets up to a real ASGI app, implement dynamic batching for throughput, and measure latency vs. throughput trade-offs.